In [2]:
import xml
import xmltodict

In [2]:
class AudioRecording():
    def __init__(self,filename,startTime):
        self.filename = filename
        self.delay = int(startTime)

class VideoRecording():
    def __init__(self,filename,startTime):
        self.filename = filename
        self.delay = int(startTime)

In [3]:
#opening xml file
with open(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\indexstream.xml') as fd:
    index = xmltodict.parse(fd.read())



In [4]:
type(index)


collections.OrderedDict

In [5]:
for item in index['root']['Message']:
    if item.get('String') and item.get('String') == 'streamAdded':
        print(item['Array']['Object']['startTime'],':',item['Array']['Object']['streamName'])


131 : /screenshare_4_2
156 : /cameraVoip_3_4


In [6]:
import pathlib
def parseindexstream(path):
    with open(str(pathlib.Path(path))) as fd:
        index = xmltodict.parse(fd.read())
    audio = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'streamAdded':
            if 'cameravoip' in str(item['Array']['Object']['streamName']).lower():
                audio.append(AudioRecording(item['Array']['Object']['streamName'],item['Array']['Object']['startTime']))
    video = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'streamAdded':
            if 'screenshare' in str(item['Array']['Object']['streamName']).lower():
                video.append(VideoRecording(item['Array']['Object']['streamName'],item['Array']['Object']['startTime']))
    return {'audio':sorted(audio,key=lambda audio:audio.delay),'video':sorted(video,key=lambda video:video.delay)}

In [7]:
result_plain_course_with_video = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\indexstream.xml'))

In [8]:
result_only_audio_course = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\only_audio_course\indexstream.xml'))



Main stream parse


In [38]:
def extract_directory(url):
    url = url.split('?download-url=')[1]
    url = url.lstrip()
    url = url.rstrip()
    url = '/'.join(url.split('/')[2:] if url.split('/')[0] == '' else url.split('/')[1:])
    url = ''.join(url.split('&name='))
    return '/'+url

def extract_filename(url):
    query = str(url.split('/')[-1]).split('&')
    for par in query:
        if 'name=' in par:
            return par.split('name=')[1]

def extract_filetype(filename):
        return filename.split('.')[-1]

class assetDocument():
    def __init__(self,assetURL,special=False):
        self.filename= extract_filename(url=assetURL) if not special else assetURL.split('/')[-1]
        self.directory = extract_directory(url=assetURL) if not special else assetURL
        self.filetype= extract_filetype(filename=self.filename)

    def __eq__(self, other):
        return self.filename == other.filename
    def __hash__(self):
        return hash(self.filename)


In [12]:

#opening xml file
with open(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\uploaded_mp4_course\mainstream.xml') as fd:
    index = xmltodict.parse(fd.read())



In [13]:
type(index)


collections.OrderedDict

In [70]:
files = []
for item in index['root']['Message']:
    if item.get('String') and item.get('String') == 'setContentSo':
        try:
            urladd = item['Array']['Object']['newValue']['documentDescriptor']['downloadUrl']
            assert urladd,'empty urladd'
        except:
            pass
        else:
            files.append(
                assetDocument(assetURL=urladd)
            )
        try:
            urladd = item['Array']['Object']['newValue']['documentDescriptor']['playbackFileNameHTMLClient']
            thename = item['Array']['Object']['newValue']['documentDescriptor']['theName']
            assert all([thename,urladd])
        except:
            continue
        else:
            print(urladd+'default/'+thename)
            files.append(
                assetDocument(assetURL=urladd+'default/'+thename,special=True)
            )

print(files)

/ptpai8lqoqls/default/Electronic_II_6_1.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/ptpai8lqoqls/default/Electronic_II_6_1.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
[<__main__.assetDocument object at 0x07839898>, <__main__.assetDocument object at 0x013C27A8>, <__main__.assetDocument object at 0x013C2028>, <__main__.assetDocument object at 0x013AEBB0>, <__main__.assetDocument object at 0x013AE658>, <__main__.assetDocument object at 0x013AED00>, <__main__.assetDocument object at 0x06E23AD8>, <__main__.assetDocument object at 0x06381358>]


In [76]:
import pathlib
def parsemainstream(path):
    with open(str(pathlib.Path(path))) as fd:
        index = xmltodict.parse(fd.read())
    assets = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'setContentSo':
            try:
                urladd = item['Array']['Object']['newValue']['documentDescriptor']['downloadUrl']
                assert urladd,'empty urladd document'
            except:
                pass
            else:
                assets.append(
                    assetDocument(assetURL=urladd)
                )
                continue
            try:
                urladd = item['Array']['Object']['newValue']['documentDescriptor']['playbackFileNameHTMLClient']
                thename = item['Array']['Object']['newValue']['documentDescriptor']['theName']
                assert all([thename,urladd])
            except:
                continue
            else:
                assets.append(
                    assetDocument(assetURL=urladd+'default/'+thename,special=True)
                )

    return list(set(assets)) if assets else None


In [78]:
m1 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\multiple_audio_video_course\mainstream.xml'))
m2 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\mainstream.xml'))
m3 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\only_audio_course\mainstream.xml'))
m4 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\uploaded_mp4_course\mainstream.xml'))


[]
[<__main__.assetDocument object at 0x011847F0>, <__main__.assetDocument object at 0x06D63598>, <__main__.assetDocument object at 0x06D635B0>, <__main__.assetDocument object at 0x06D63658>, <__main__.assetDocument object at 0x06D63580>]
/ptpai8lqoqls/default/Electronic_II_6_1.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/ptpai8lqoqls/default/Electronic_II_6_1.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
/pzcmvjg80cbt/default/Electronic_II_6_2.mp4
[<__main__.assetDocument object at 0x074E7B50>, <__main__.assetDocument object at 0x01184070>, <__main__.assetDocument object at 0x0637B2F8>, <__main__.assetDocument object at 0x078463B8>, <__main__.assetDocument object at 0x07846E38>, <__main__.assetDocument object at 0x013C2388>, <__main__.assetDocument object at 0x013C26D0>, <__main__.assetDocument object at 0x013C2B68>]
